Feature engineering using **Titanic** dataset
**Medium link that involves detailed description for Feature Engineering**
[https://medium.com/@grkemaksoy/feature-engineering-4cb941f6d875](http://)

4 different subtopics in Feature Engineering: 

# Outliers
# Missing Values
# Encoding/Scaling (Will be in Part 2)
# Feature extraction (Will be in Part 2)


In [1]:
import numpy as np
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
titanic = pd.read_csv("/kaggle/input/titanic/train.csv")

In [2]:
#Step 1: Detecting numerical and categorical columns 
cat_col = [col for col in titanic.columns if titanic[col].dtype == "O"]
cat_col
num_col = [col for col in titanic.columns if titanic[col].dtype != "O"]
num_col
num_but_cat = [col for col in num_col if titanic[col].nunique() < 10]
num_col = [col for col in num_col if col not in num_but_cat]

#Passenger Id is actually not a numerical variable. Drop it from num_col
num_col = [col for col in num_col if col != "PassengerId"]
num_but_cat
num_col

#We detect cardinal variables which has more than 20 unique values
cat_but_car = [col for col in cat_col if titanic[col].nunique() >20]
cat_col = [col for col in cat_col if col not in cat_but_car]
for col in num_but_cat: cat_col.append(col)
cat_but_car
cat_col

['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']

['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

['Survived', 'Pclass', 'SibSp', 'Parch']

['Age', 'Fare']

['Name', 'Ticket', 'Cabin']

['Sex', 'Embarked', 'Survived', 'Pclass', 'SibSp', 'Parch']

In [3]:
#defining a function for detecting variable types
def categories(dataframe):
    cat_col = [col for col in dataframe.columns if dataframe[col].dtype == "O" 
               or dataframe[col].nunique()<10]
    num_col = [col for col in dataframe.columns if dataframe[col].dtype != "O" 
               and dataframe[col].nunique()>10]
    num_col = [col for col in dataframe.columns if dataframe[col].dtype != "O" 
               and dataframe[col].nunique()>10 and col != "PassengerId"]
    cat_but_car = [col for col in cat_col if dataframe[col].nunique() >20]
    
    return cat_col, num_col, cat_but_car

#assigning variable names
cat_col, num_col, cat_but_car = categories(titanic)
display(cat_col, num_col, cat_but_car)

['Survived',
 'Pclass',
 'Name',
 'Sex',
 'SibSp',
 'Parch',
 'Ticket',
 'Cabin',
 'Embarked']

['Age', 'Fare']

['Name', 'Ticket', 'Cabin']

# OUTLIERS

In [4]:
#Step 2: Observing percentiles for numerical columns
for col in num_col: 
    display(titanic[col].describe([.05, .10, .25, .50, .75, .90, .95]).T)

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
5%         4.000000
10%       14.000000
25%       20.125000
50%       28.000000
75%       38.000000
90%       50.000000
95%       56.000000
max       80.000000
Name: Age, dtype: float64

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
5%         7.225000
10%        7.550000
25%        7.910400
50%       14.454200
75%       31.000000
90%       77.958300
95%      112.079150
max      512.329200
Name: Fare, dtype: float64

In [5]:
#Step 2.1: Outlier detection using 25th and 75th percentiles. Since we have not many 
#observations, number of observations that detected as outlier can cause troubles 
#such as loss of information. 
def outlier_thresholds(dataframe, col, low= 0.25, high = 0.75):
    lim_l = dataframe[col].quantile(low)
    lim_h = dataframe[col].quantile(high)
    iqr = lim_h - lim_l
    upper = lim_h + 1.5*iqr
    lower = lim_l - 1.5*iqr
    return lower, upper

age_lower, age_upper = outlier_thresholds(titanic, "Age")
display (age_lower, age_upper)

-6.6875

64.8125

In [6]:
#Step 2.2: Function detecting index numbers of outliers 
def outlier_indices(dataframe, col, get_index = False):
    low, up = outlier_thresholds(dataframe, col)
    #Print all outliers if number of outlier is below 10, else print only first 5 of
    #them
    if dataframe.loc[(dataframe[col]<low) | (dataframe[col]>up)].shape[0] >10:
        print(dataframe.loc[(dataframe[col]<low) | (dataframe[col]>up)].head())
    else:
        print(dataframe.loc[(dataframe[col]<low) | (dataframe[col]>up)])
    if get_index:
        indices = dataframe.loc[(dataframe[col]<low) | (dataframe[col]>up)].index
        return indices

outlier_indices(titanic, "Age", get_index=True)

     PassengerId  Survived  Pclass                            Name   Sex  \
33            34         0       2           Wheadon, Mr. Edward H  male   
54            55         0       1  Ostby, Mr. Engelhart Cornelius  male   
96            97         0       1       Goldschmidt, Mr. George B  male   
116          117         0       3            Connors, Mr. Patrick  male   
280          281         0       3                Duane, Mr. Frank  male   

      Age  SibSp  Parch      Ticket     Fare Cabin Embarked  
33   66.0      0      0  C.A. 24579  10.5000   NaN        S  
54   65.0      0      1      113509  61.9792   B30        C  
96   71.0      0      0    PC 17754  34.6542    A5        C  
116  70.5      0      0      370369   7.7500   NaN        Q  
280  65.0      0      0      336439   7.7500   NaN        Q  


Int64Index([33, 54, 96, 116, 280, 456, 493, 630, 672, 745, 851], dtype='int64')

In [7]:
#Step 2.3: Handling outliers
#a- Drop all outliers
def remove_outliers(dataframe, col):
    low, up = outlier_thresholds(dataframe, col)
    df_remove_outlier = dataframe.loc[(dataframe[col]>low) & (dataframe[col]<up)]
    return df_remove_outlier

df_wo_outliers = remove_outliers(titanic, "Age")
df_wo_outliers.shape

(703, 12)

In [8]:
#b- Assign lower and upper threshold values to outliers
def reassign_outliers(dataframe, col):
    low, up = outlier_thresholds(dataframe, col)
    df_reassigned_outliers = dataframe
    df_reassigned_outliers.loc[df_reassigned_outliers[col]<low, col] = low
    df_reassigned_outliers.loc[df_reassigned_outliers[col]>up, col] = up
    return df_reassigned_outliers

df_reassigned_outliers = reassign_outliers(titanic, "Age")
df_reassigned_outliers[df_reassigned_outliers["Age"] > 64]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
33,34,0,2,"Wheadon, Mr. Edward H",male,64.8125,0,0,C.A. 24579,10.5000,NaN,S
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,64.8125,0,1,113509,61.9792,B30,C
96,97,0,1,"Goldschmidt, Mr. George B",male,64.8125,0,0,PC 17754,34.6542,A5,C
116,117,0,3,"Connors, Mr. Patrick",male,64.8125,0,0,370369,7.7500,NaN,Q
280,281,0,3,"Duane, Mr. Frank",male,64.8125,0,0,336439,7.7500,NaN,Q
456,457,0,1,"Millet, Mr. Francis Davis",male,64.8125,0,0,13509,26.5500,E38,S
493,494,0,1,"Artagaveytia, Mr. Ramon",male,64.8125,0,0,PC 17609,49.5042,NaN,C
630,631,1,1,"Barkworth, Mr. Algernon Henry Wilson",male,64.8125,0,0,27042,30.0000,A23,S
672,673,0,2,"Mitchell, Mr. Henry Michael",male,64.8125,0,0,C.A. 24580,10.5000,NaN,S
745,746,0,1,"Crosby, Capt. Edward Gifford",male,64.8125,1,1,WE/P 5735,71.0000,B22,S


In [9]:
#c- Local Outlier Factor - Using all numerical variables to detect outliers as a whole
#rather than observing them for each variables. 
from sklearn.neighbors import LocalOutlierFactor
def local_outlier_factor(dataframe, n_neighbors = 20):
    dataframe = dataframe.select_dtypes(include=['float64', 'int64'])
    dataframe = dataframe.dropna()
    clf = LocalOutlierFactor(n_neighbors = n_neighbors)
    clf.fit_predict(dataframe)
    df_scores = clf.negative_outlier_factor_
    th = np.sort(df_scores)[3]
    print(dataframe[df_scores < th].index)
    lof_df = dataframe[df_scores < th].drop(axis=0, labels=dataframe[df_scores < th].index)
    return lof_df

lof_titanic = local_outlier_factor(titanic)

Int64Index([27, 679, 737], dtype='int64')


# MISSING VALUES

In [10]:
# Step 3: Detecting Missing Value
display(titanic.isnull().values.any())
#Number of missing values for each variables in descending order
display(titanic.isnull().sum().sort_values(ascending=False))

True

Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Fare             0
dtype: int64

In [11]:
#Step 3.1: Missing value ratios
def missing_values(dataframe):
    values = dataframe.isnull().sum().sort_values(ascending=False)
    perc = (dataframe.isnull().sum().sort_values(ascending=False) / titanic.shape[0])*100
    table = pd.concat([values, np.round(perc, 2)], axis=1, keys=["# of missing values", "% of missing values"])
    return table

missing_summary = missing_values(titanic)
missing_summary

,# of missing values,% of missing values
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
SibSp,0,0.00
Parch,0,0.00


In [12]:
#Step 3.2: Handling Missing Values
#a-Drop all of the missing values
titanic.dropna()
#b-Assigning mean, median for numerical variables, mode for categorical variables
titanic["Age"].fillna(titanic["Age"].mean())
titanic["Age"].fillna(titanic["Age"].median())
titanic["Cabin"].fillna(titanic["Cabin"].mode())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
...,...,...,...,...,...,...,...,...,...,...,...,...
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
872,873,0,1,"Carlsson, Mr. Frans Olof",male,33.0,0,0,695,5.0000,B51 B53 B55,S
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S


0      22.0000
1      38.0000
2      26.0000
3      35.0000
4      35.0000
        ...   
886    27.0000
887    19.0000
888    29.6227
889    26.0000
890    32.0000
Name: Age, Length: 891, dtype: float64

0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888    28.0
889    26.0
890    32.0
Name: Age, Length: 891, dtype: float64

0      B96 B98
1          C85
2           G6
3         C123
4          NaN
        ...   
886        NaN
887        B42
888        NaN
889       C148
890        NaN
Name: Cabin, Length: 891, dtype: object

In [13]:
#c-Assigning values with respect to different categories (Using sex for assigning 
#missing variables of age)
#Alternative 1
titanic["Age"].fillna(titanic.groupby("Sex")["Age"].transform("mean"))
#Alternative 2
titanic.loc[(titanic["Age"].isnull()) & (titanic["Sex"]=="female"), "Age"] = titanic.groupby("Sex") \
["Age"].mean()["female"]
titanic.loc[(titanic["Age"].isnull()) & (titanic["Sex"]=="male"), "Age"] = titanic.groupby("Sex") \
["Age"].mean()["male"]

0      22.000000
1      38.000000
2      26.000000
3      35.000000
4      35.000000
         ...    
886    27.000000
887    19.000000
888    27.915709
889    26.000000
890    32.000000
Name: Age, Length: 891, dtype: float64

In [14]:
#d-Assigning values with respect to nearest neighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
cat_col, num_col, cat_but_car = categories(titanic)
#No need to include "Name" and "Ticket" variables
cat_col = [col for col in cat_col if col != "Name" and col != "Ticket"]
titanic_missing_pred = pd.get_dummies(titanic[cat_col + num_col], drop_first=True)

#In order to use KNN, we first need to standardize numerical columns
scaler = MinMaxScaler()
titanic_missing_pred = pd.DataFrame(scaler.fit_transform(titanic_missing_pred), 
                                    columns=titanic_missing_pred.columns)
imputer = KNNImputer(n_neighbors=5)
titanic_missing_pred = pd.DataFrame(imputer.fit_transform(titanic_missing_pred), 
                                    columns=titanic_missing_pred.columns)
display(titanic_missing_pred.head())
#After assigning values, we convert the numerical variables to their first values
#using inverse transfomr method
titanic_missing_pred = pd.DataFrame(scaler.inverse_transform(titanic_missing_pred), 
                                    columns=titanic_missing_pred.columns)

titanic["age_imputed_knn"] = titanic_missing_pred[["Age"]]
display(titanic.loc[titanic["Age"].isnull(),["Age", "age_imputed_knn"]])

,Survived,Pclass,SibSp,Parch,Age,Fare,Sex_male,Cabin_A14,Cabin_A16,Cabin_A19,...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,0.0,1.0,0.125,0.0,0.335132,0.014151,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.125,0.0,0.583608,0.139136,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.000,0.0,0.397251,0.015469,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.125,0.0,0.537019,0.103644,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.000,0.0,0.537019,0.015713,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,Age,age_imputed_knn
